In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/rabieelkharoua/predicting-manufacturing-defects-dataset/manufacturing_defect_dataset.csv


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_auc_score, precision_recall_curve, auc)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE

# Boosting models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [3]:
data = pd.read_csv("/kaggle/input/datasets/rabieelkharoua/predicting-manufacturing-defects-dataset/manufacturing_defect_dataset.csv")

data.fillna(0, inplace=True)

X = data.drop('DefectStatus', axis=1)
y = data['DefectStatus']

In [4]:
selector = SelectKBest(score_func=chi2, k=10)
X_selected = selector.fit_transform(X, y)

selected_features = X.columns[selector.get_support()]
print("Selected Features:", list(selected_features))

Selected Features: ['ProductionVolume', 'ProductionCost', 'SupplierQuality', 'DefectRate', 'QualityScore', 'MaintenanceHours', 'SafetyIncidents', 'EnergyConsumption', 'EnergyEfficiency', 'AdditiveMaterialCost']


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, stratify=y, random_state=42
)

In [6]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [7]:
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

In [8]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    
    "XGBoost": XGBClassifier(eval_metric='logloss', use_label_encoder=False)
}

In [9]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    precision, recall, _ = precision_recall_curve(y_test, y_prob)
    pr_auc = auc(recall, precision)

    print(f"\n===== {name} =====")
    print("Accuracy:", acc)
    print("ROC-AUC:", roc)
    print("PR-AUC:", pr_auc)
    print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

    return model

In [10]:
trained_models = {}

for name, model in models.items():
    trained_models[name] = evaluate_model(name, model,
                                          X_train_res, y_train_res,
                                          X_test_scaled, y_test)


===== Logistic Regression =====
Accuracy: 0.7546296296296297
ROC-AUC: 0.7941569430836377
PR-AUC: 0.9330827715020478

Confusion Matrix:
 [[ 76  27]
 [132 413]]

Classification Report:
               precision    recall  f1-score   support

           0       0.37      0.74      0.49       103
           1       0.94      0.76      0.84       545

    accuracy                           0.75       648
   macro avg       0.65      0.75      0.66       648
weighted avg       0.85      0.75      0.78       648


===== Random Forest =====
Accuracy: 0.9475308641975309
ROC-AUC: 0.8396187761646032
PR-AUC: 0.9435118753532954

Confusion Matrix:
 [[ 75  28]
 [  6 539]]

Classification Report:
               precision    recall  f1-score   support

           0       0.93      0.73      0.82       103
           1       0.95      0.99      0.97       545

    accuracy                           0.95       648
   macro avg       0.94      0.86      0.89       648
weighted avg       0.95      0.95    

In [11]:
print("\n===== CROSS VALIDATION SCORES =====")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train_res, y_train_res,
                             cv=cv, scoring='roc_auc')
    print(f"{name}: Mean ROC-AUC = {scores.mean():.4f}")



===== CROSS VALIDATION SCORES =====
Logistic Regression: Mean ROC-AUC = 0.8668
Random Forest: Mean ROC-AUC = 0.9906
XGBoost: Mean ROC-AUC = 0.9920


In [12]:
rf_model = trained_models["Random Forest"]

importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print("\n===== FEATURE IMPORTANCE =====")
print(feature_importance_df)


===== FEATURE IMPORTANCE =====
                Feature  Importance
5      MaintenanceHours    0.299874
3            DefectRate    0.221792
4          QualityScore    0.158687
0      ProductionVolume    0.111246
9  AdditiveMaterialCost    0.038517
7     EnergyConsumption    0.037468
2       SupplierQuality    0.036936
1        ProductionCost    0.035803
8      EnergyEfficiency    0.032717
6       SafetyIncidents    0.026960


In [14]:
perm_importance = permutation_importance(
    rf_model, X_test_scaled, y_test, n_repeats=10, random_state=42
)

perm_df = pd.DataFrame({
    "Feature": selected_features,
    "Importance": perm_importance.importances_mean
}).sort_values(by="Importance", ascending=False)

print("\n===== PERMUTATION IMPORTANCE =====")
print(perm_df)


===== PERMUTATION IMPORTANCE =====
                Feature  Importance
5      MaintenanceHours    0.120062
3            DefectRate    0.093673
4          QualityScore    0.084105
0      ProductionVolume    0.053086
7     EnergyConsumption    0.000617
9  AdditiveMaterialCost    0.000463
1        ProductionCost    0.000309
8      EnergyEfficiency    0.000309
2       SupplierQuality    0.000000
6       SafetyIncidents   -0.000463


In [16]:
ensemble = VotingClassifier(
    estimators=[
        ('rf', RandomForestClassifier(n_estimators=100)),
        ('xgb', XGBClassifier(eval_metric='logloss'))
    ],
    voting='soft'
)

print("\n===== ENSEMBLE MODEL =====")
evaluate_model("Voting Ensemble",
               ensemble,
               X_train_res, y_train_res,
               X_test_scaled, y_test)


===== ENSEMBLE MODEL =====

===== Voting Ensemble =====
Accuracy: 0.9398148148148148
ROC-AUC: 0.8318161574775096
PR-AUC: 0.9341596566372338

Confusion Matrix:
 [[ 75  28]
 [ 11 534]]

Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.73      0.79       103
           1       0.95      0.98      0.96       545

    accuracy                           0.94       648
   macro avg       0.91      0.85      0.88       648
weighted avg       0.94      0.94      0.94       648



VotingClassifier(estimators=[('rf', RandomForestClassifier()),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None,
                                            feature_weights=None, gamma=None,
                                            grow_policy=None,
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None, ...))],
                 voting='soft')